# LFW Grad-CAM — 00. Source and model freeze

Grad-CAM보다 먼저 완료된 Step 2 정량 결과, 고정 crop manifest,
ModelSpec을 SHA-256으로 동결합니다. Step 1/Step 2 정량 artifact는
수정하지 않으며 fallback이 사용된 결과를 입력으로 허용하지 않습니다.

In [ ]:
from __future__ import annotations

from pathlib import Path
import sys

import yaml

PROJECT_ROOT = Path.cwd().resolve()
for candidate in (PROJECT_ROOT, *PROJECT_ROOT.parents):
    if (candidate / "research").is_dir() and (candidate / "configs").is_dir():
        PROJECT_ROOT = candidate
        break
else:
    raise RuntimeError("프로젝트 루트(D:/ronbun)를 찾을 수 없습니다.")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

CONFIG_PATH = PROJECT_ROOT / "configs/experiments/step2_pytorch_gradcam.yaml"
CONFIG = yaml.safe_load(CONFIG_PATH.read_text(encoding="utf-8"))

MODEL_NAME = "arcface"     # "arcface", "adaface", "magface" 중 이번 실행 모델
MODE = "dev"               # 빠른 검증은 "dev", 전체 논문 실행만 "real"
DATA_FRACTION = 0.10       # identity 단위 사용 비율; 0 < 값 <= 1
SEED = 42                  # 부분집합·tie-break·random control 재현 seed
EXECUTE_STAGE = False      # 입력과 checkpoint를 채운 뒤 실제 계산할 때만 True
WRITE_OUTPUTS = False      # 검증 후 새 artifact를 저장할 때만 True

if MODEL_NAME not in CONFIG["models"]["selected"]:
    raise ValueError(f"지원하지 않는 모델: {MODEL_NAME}")
if MODE not in {"dev", "real"}:
    raise ValueError("MODE는 'dev' 또는 'real'이어야 합니다.")
if not 0.0 < DATA_FRACTION <= 1.0:
    raise ValueError("DATA_FRACTION은 (0, 1] 범위여야 합니다.")
if WRITE_OUTPUTS and not EXECUTE_STAGE:
    raise ValueError("WRITE_OUTPUTS=True이면 EXECUTE_STAGE도 True여야 합니다.")

In [ ]:
import json
import pandas as pd

from research.embeddings import read_model_spec
from research.runtime.hashing import sha256_file

MODEL_SPEC_PATH = None
PAIRED_METRICS_PATH = None
RETRIEVAL_METRICS_PATH = None
ALIGNED_CROP_MANIFEST_PATH = None
FREEZE_MANIFEST_PATH = None       # 새 run 안의 JSON 경로

In [ ]:
inputs = {
    "model_spec": MODEL_SPEC_PATH,
    "paired_metrics": PAIRED_METRICS_PATH,
    "retrieval_metrics": RETRIEVAL_METRICS_PATH,
    "aligned_crop_manifest": ALIGNED_CROP_MANIFEST_PATH,
}
if EXECUTE_STAGE:
    missing = [name for name, value in inputs.items() if value is None]
    if missing:
        raise RuntimeError(f"입력 경로가 비어 있습니다: {missing}")
    resolved = {name: Path(path).resolve() for name, path in inputs.items()}
    for name, path in resolved.items():
        if not path.is_file():
            raise FileNotFoundError(f"{name}: {path}")
    spec = read_model_spec(resolved["model_spec"], verify_checkpoint=True)
    if spec.family != MODEL_NAME:
        raise ValueError("MODEL_NAME과 ModelSpec family가 다릅니다.")

    paired = pd.read_parquet(resolved["paired_metrics"])
    retrieval = pd.read_parquet(resolved["retrieval_metrics"])
    for name, frame in (("paired", paired), ("retrieval", retrieval)):
        if "origin_fallback_used" not in frame:
            raise ValueError(f"{name} 결과에 origin_fallback_used가 없습니다.")
        if frame["origin_fallback_used"].fillna(True).astype(bool).any():
            raise ValueError(f"{name} 결과에 origin fallback 사용 행이 있습니다.")
        if "model_uid" in frame and set(frame["model_uid"]) != {spec.model_uid}:
            raise ValueError(f"{name} 결과의 model_uid가 ModelSpec과 다릅니다.")

    freeze_manifest = {
        "dataset": "lfw",
        "model_uid": spec.model_uid,
        "checkpoint_sha256": spec.checkpoint.sha256,
        "preprocess_hash": spec.preprocessing.preprocess_hash,
        "config_sha256": sha256_file(CONFIG_PATH),
        "inputs": {
            name: {"path": str(path), "sha256": sha256_file(path)}
            for name, path in resolved.items()
        },
        "fallback_free": True,
    }
    if WRITE_OUTPUTS:
        if FREEZE_MANIFEST_PATH is None:
            raise RuntimeError("FREEZE_MANIFEST_PATH를 지정하세요.")
        destination = Path(FREEZE_MANIFEST_PATH).resolve()
        if destination.exists():
            raise FileExistsError(f"기존 freeze를 덮어쓸 수 없습니다: {destination}")
        destination.parent.mkdir(parents=True, exist_ok=True)
        destination.write_text(
            json.dumps(freeze_manifest, ensure_ascii=False, indent=2),
            encoding="utf-8",
        )
else:
    freeze_manifest = {
        "status": "not_executed",
        "reason": "EXECUTE_STAGE=False",
    }
freeze_manifest

hash가 하나라도 바뀌면 기존 Grad-CAM run을 이어 쓰지 말고 새 run에서
00부터 다시 시작합니다.